In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from tqdm import tqdm
import math
import numpy as np
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter

In [2]:

# 实验数据（示例）
theta1 = torch.tensor([0.1, 0.2, 0.3, 0.4], dtype=torch.float32)  # 输入 theta1
theta2 = torch.tensor([0.5, 0.6, 0.7, 0.8], dtype=torch.float32)  # 输入 theta2
P_actual = torch.tensor([10.5, 12.3, 14.7, 18.2], dtype=torch.float32)  # 实验输出 P

# 定义力学模型
class MechanicsModel(nn.Module):
    def __init__(self):
        super(MechanicsModel, self).__init__()
        # 初始化待辨识参数 (k, b)
        self.k = nn.Parameter(torch.tensor(1.0, dtype=torch.float32))  # 刚度参数
        self.b = nn.Parameter(torch.tensor(0.1, dtype=torch.float32))  # 阻尼参数

    def forward(self, theta1, theta2):
        # 力学模型公式 (示例)
        P_pred = self.k * theta1 + self.b * theta2
        return P_pred

# 实例化模型
model = MechanicsModel()

# 定义损失函数（均方误差）
loss_fn = nn.MSELoss()

# 定义优化器（Adam 优化器）
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 训练模型
num_epochs = 50000
for epoch in range(num_epochs):
    # 前向传播
    P_pred = model(theta1, theta2)
    loss = loss_fn(P_pred, P_actual)

    # 反向传播和优化
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # 每隔 100 轮打印一次损失
    if (epoch + 1) % 10000 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# 输出辨识的参数
print(f"Estimated k: {model.k.item():.4f}")
print(f"Estimated b: {model.b.item():.4f}")

print(model.k * theta1 + model.b * theta2 - P_actual)

Epoch [10000/50000], Loss: 0.1870
Epoch [20000/50000], Loss: 0.1838
Epoch [30000/50000], Loss: 0.1838
Epoch [40000/50000], Loss: 0.1838
Epoch [50000/50000], Loss: 0.1838
Estimated k: 6.6250
Estimated b: 18.8750
tensor([-0.4000,  0.3500,  0.5000, -0.4500], grad_fn=<SubBackward0>)


In [3]:
data = pd.read_csv("../../log/realdata/StaticProcess/StaticPoint_6group.csv")
print(type(data))

print(data)
# 实验数据
P1_array = data['P1'].values
P2_array = data['P2'].values
theta1_array = data['theta1'].values
theta2_array = data['theta2'].values

# 实验数据
theta1 = torch.tensor(theta1_array, dtype=torch.float32)*torch.pi/180  # 输入 theta1
theta2 = torch.tensor(theta2_array, dtype=torch.float32)*torch.pi/180  # 输入 theta2
P1_actual = torch.tensor(P1_array, dtype=torch.float32)  # 实验输出 P
P2_actual = torch.tensor(P2_array, dtype=torch.float32)  # 实验输出 P
P_actual = torch.stack([P1_actual, P2_actual], dim=1)
P_actual = P_actual*1000
print(P_actual)

<class 'pandas.core.frame.DataFrame'>
      P1    P2      theta1     theta2
0    0.0   0.0   73.892669  62.259572
1    0.0  10.0   74.028660  46.373011
2    0.0  20.0   74.099886  28.347963
3    0.0  30.0   74.415763  15.721740
4   10.0   0.0   83.981882  51.382921
5   10.0  10.0   83.774517  36.056016
6   10.0  20.0   84.417009  12.798675
7   20.0   0.0   92.744447  41.745600
8   20.0  10.0   92.879490  26.483622
9   30.0   0.0   99.265285  34.066376
10  30.0  10.0  101.892863  16.289084
11  40.0   0.0  110.089712  20.816638
12  50.0   0.0  118.192064   9.778827
tensor([[    0.,     0.],
        [    0., 10000.],
        [    0., 20000.],
        [    0., 30000.],
        [10000.,     0.],
        [10000., 10000.],
        [10000., 20000.],
        [20000.,     0.],
        [20000., 10000.],
        [30000.,     0.],
        [30000., 10000.],
        [40000.,     0.],
        [50000.,     0.]])


In [17]:
# 定义力学模型
class MechanicsModel(nn.Module):
    def __init__(self):
        super(MechanicsModel, self).__init__()
        # 初始化待辨识参数：k_3, k_4, m_1, m_2, m_3, m_4, l_10, l_20, S_1, S_2
        self.k_3 = nn.Parameter(torch.tensor(300, dtype=torch.float32))
        self.k_4 = nn.Parameter(torch.tensor(300, dtype=torch.float32))
        self.m_1 = nn.Parameter(torch.tensor(0.086, dtype=torch.float32))
        self.m_2 = nn.Parameter(torch.tensor(0.1033, dtype=torch.float32))
        self.m_3 = nn.Parameter(torch.tensor(186.48 * 1e-3, dtype=torch.float32))
        self.m_4 = nn.Parameter(torch.tensor(272.66 * 1e-3, dtype=torch.float32))
        self.l_10 = nn.Parameter(torch.tensor(0.174, dtype=torch.float32))
        self.l_20 = nn.Parameter(torch.tensor(0.252, dtype=torch.float32))
        self.S_1 = nn.Parameter(torch.tensor(0.0006, dtype=torch.float32))
        self.S_2 = nn.Parameter(torch.tensor(0.0006, dtype=torch.float32))

        
    def forward(self, theta_1, theta_2):
        # 静力学模型公式        
        # 固定参数
        a_1, a_2, b_1, b_2, d_1, d_2 = 0.25, 0.25, 0.21213, 0.1, 0.06, 0.10
        beta_1, beta_2 = 8.13 / 180 * torch.pi, 30 / 180 * torch.pi
        g = 9.8

        A_x_O = d_1
        A_y_O = 0

        B_x_O = -d_2
        B_y_O = 0

        C_x_O = b_1 * torch.cos(theta_1 - beta_1)
        C_y_O = b_1 * torch.sin(theta_1 - beta_1)

        D_x_O = a_1 * torch.cos(theta_1) + b_2 * torch.cos(theta_1 + theta_2 + beta_2)
        D_y_O = a_1 * torch.sin(theta_1) + b_2 * torch.sin(theta_1 + theta_2 + beta_2)

        # E_x_O = a_1 * torch.cos(theta_1)
        # E_y_O = a_1 * torch.sin(theta_1)

        # F_x_O = a_1 * torch.cos(theta_1) + a_2 * torch.cos(theta_1 + theta_2)
        # F_y_O = a_1 * torch.sin(theta_1) + a_2 * torch.sin(theta_1 + theta_2)

        # 计算偏导数
        # 对 theta_1 的偏导数

        dA_x_O_dtheta_1 = 0
        dA_y_O_dtheta_1 = 0

        dB_x_O_dtheta_1 = 0
        dB_y_O_dtheta_1 = 0

        dC_x_O_dtheta_1 = -b_1 * torch.sin(theta_1 - beta_1)
        dC_y_O_dtheta_1 = b_1 * torch.cos(theta_1 - beta_1)

        dD_x_O_dtheta_1 = -a_1 * torch.sin(theta_1) - b_2 * torch.sin(theta_1 + theta_2 + beta_2)
        dD_y_O_dtheta_1 = a_1 * torch.cos(theta_1) + b_2 * torch.cos(theta_1 + theta_2 + beta_2)

        dE_x_O_dtheta_1 = -a_1 * torch.sin(theta_1)
        dE_y_O_dtheta_1 = a_1 * torch.cos(theta_1)

        dF_x_O_dtheta_1 = -a_1 * torch.sin(theta_1) - a_2 * torch.sin(theta_1 + theta_2)
        dF_y_O_dtheta_1 = a_1 * torch.cos(theta_1) + a_2 * torch.cos(theta_1 + theta_2)

        # 对 theta_2 的偏导数

        dA_x_O_dtheta_2 = 0
        dA_y_O_dtheta_2 = 0

        dB_x_O_dtheta_2 = 0
        dB_y_O_dtheta_2 = 0

        dC_x_O_dtheta_2 = 0
        dC_y_O_dtheta_2 = 0

        dD_x_O_dtheta_2 = -b_2 * torch.sin(theta_1 + theta_2 + beta_2)
        dD_y_O_dtheta_2 = b_2 * torch.cos(theta_1 + theta_2 + beta_2)

        dE_x_O_dtheta_2 = 0
        dE_y_O_dtheta_2 = 0

        dF_x_O_dtheta_2 = -a_2 * torch.sin(theta_1 + theta_2)
        dF_y_O_dtheta_2 = a_2 * torch.cos(theta_1 + theta_2)

        # 计算长度 l1 和 l2
        l_1 = torch.sqrt((A_x_O - C_x_O)**2 + (A_y_O - C_y_O)**2)
        l_2 = torch.sqrt((B_x_O - D_x_O)**2 + (B_y_O - D_y_O)**2)

        # 计算偏导数
        # 偏导数 d/dtheta_1
        dl_1_dtheta_1 = 1/l_1 * ((A_x_O - C_x_O) * (dA_x_O_dtheta_1 - dC_x_O_dtheta_1) + (A_y_O - C_y_O) * (dA_y_O_dtheta_1 - dC_y_O_dtheta_1))
        dl_2_dtheta_1 = 1/l_2 * ((B_x_O - D_x_O) * (dB_x_O_dtheta_1 - dD_x_O_dtheta_1) + (B_y_O - D_y_O) * (dB_y_O_dtheta_1 - dD_y_O_dtheta_1))

        # 偏导数 d/dtheta_2
        dl_1_dtheta_2 = 1/l_1 * ((A_x_O - C_x_O) * (dA_x_O_dtheta_2 - dC_x_O_dtheta_2) + (A_y_O - C_y_O) * (dA_y_O_dtheta_2 - dC_y_O_dtheta_2))
        dl_2_dtheta_2 = 1/l_2 * ((B_x_O - D_x_O) * (dB_x_O_dtheta_2 - dD_x_O_dtheta_2) + (B_y_O - D_y_O) * (dB_y_O_dtheta_2 - dD_y_O_dtheta_2))

        # print(dl_1_dtheta_2)  # check the model

        # 等式右侧
        RHSb_1 = -(self.m_1*g*(dE_y_O_dtheta_1/2) + self.m_2*g*((dE_y_O_dtheta_1+dF_y_O_dtheta_1)/2) + self.m_3*g*(dC_y_O_dtheta_1/2) + self.m_4*g*(dD_y_O_dtheta_1/2) )
        RHSb_2 = -(self.m_1*g*(dE_y_O_dtheta_2/2) + self.m_2*g*((dE_y_O_dtheta_2+dF_y_O_dtheta_2)/2) + self.m_3*g*(dC_y_O_dtheta_2/2) + self.m_4*g*(dD_y_O_dtheta_2/2) )
        b = torch.stack([RHSb_1, RHSb_2], dim=1)      # 注意，不能使用tensor创建，GPT推荐使用torch.stack
        # print(b.shape)

        # 等式左侧
        LHSA = torch.stack([
            torch.stack([dl_1_dtheta_1, dl_2_dtheta_1], dim=1), 
            torch.stack([dl_1_dtheta_2, dl_2_dtheta_2], dim=1)
            ], dim=1)

        # 回复力
        F_k = torch.stack([self.k_3*(l_1-self.l_10), self.k_4*(l_2-self.l_20)], dim=1)

        # 计算静力学
        StaticForce = torch.linalg.solve(LHSA, b) + F_k
        StaticP_pred = torch.stack([StaticForce[:, 0]/self.S_1, StaticForce[:, 1]/self.S_2], dim=1)
        # 归一化写法
        # StaticP_pred = StaticForce # torch.stack([StaticForce[:, 0]*torch.exp(self.S_1), StaticForce[:, 1]*torch.exp(self.S_2)], dim=1)
        return StaticP_pred



In [19]:
# 检验测试数据
model_test = MechanicsModel()
model_test.k_3.data = torch.tensor(142.22034830974255, dtype=torch.float32)
model_test.k_4.data = torch.tensor(129.49423449130097, dtype=torch.float32)
model_test.m_1.data = torch.tensor(0.086, dtype=torch.float32)
model_test.m_2.data = torch.tensor(0.1033, dtype=torch.float32)
model_test.m_3.data = torch.tensor(186.48 * 1e-3, dtype=torch.float32)
model_test.m_4.data = torch.tensor(272.66 * 1e-3, dtype=torch.float32)
model_test.l_10.data = torch.tensor(0.1641527577313371, dtype=torch.float32)
model_test.l_20.data = torch.tensor(0.2579251802483194, dtype=torch.float32)
model_test.S_1.data = torch.tensor(0.0003490837132109409, dtype=torch.float32)
model_test.S_2.data = torch.tensor(0.0003815501584986775, dtype=torch.float32)
# model_test.eval()
P_pred = model_test(theta1, theta2)
loss_fn = nn.MSELoss()
loss = loss_fn(P_pred, P_actual)
print(math.sqrt(loss)/1000)



0.9389302423503038


In [ ]:
# 制造数据
# 实例化模型
model = MechanicsModel()
model.eval()
P_pred = model(theta1, theta2)
# list all experiments
for i in range(len(theta1)):
    print(f"theta1: {theta1[i]:.4f}, theta2: {theta2[i]:.4f}, P_actual: {P_actual[i]}, P_pred: {P_pred[i]}")
# 初始误差
# 定义损失函数（均方误差）
loss_fn = nn.MSELoss()
loss = loss_fn(P_pred, P_actual)
print(f"Initial Loss: {math.sqrt(loss.item())/1000:.4f}")


In [ ]:
# 实例化模型
model = MechanicsModel()

# 设置不优化的参数
model.m_1.requires_grad = False
model.m_2.requires_grad = False
model.m_3.requires_grad = False
model.m_4.requires_grad = False
model.l_10.requires_grad = False
model.l_20.requires_grad = False

# 定义损失函数（均方误差）
loss_fn = nn.MSELoss()

# 定义优化器（Adam 优化器）
optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)   # 使用 L2 正则化

# 创建 TensorBoard 的 SummaryWriter
# 使用时间戳创建唯一的日志文件夹
log_dir = f"runs/mechanics_model_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
writer = SummaryWriter(log_dir=log_dir)

# 训练模型
num_epochs = 50000 * 10
for epoch in tqdm(range(num_epochs)):
    # 前向传播
    P_pred = model(theta1, theta2)
    loss = loss_fn(P_pred, P_actual)

    # 反向传播和优化
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # 每隔 10000 轮打印一次损失
    if (epoch + 1) % 10000 == 0:

        # 记录损失值
        writer.add_scalar("Loss/train", math.sqrt(loss.item()) / 1000, epoch + 1)
        
        # 记录参数值
        writer.add_scalar("Params/S_1", torch.exp(-model.S_1).item(), epoch + 1)
        writer.add_scalar("Params/S_2", torch.exp(-model.S_2).item(), epoch + 1)
        writer.add_scalar("Params/k_3", model.k_3.item(), epoch + 1)
        writer.add_scalar("Params/k_4", model.k_4.item(), epoch + 1)
        writer.add_scalar("Params/l_10", model.l_10.item(), epoch + 1)
        writer.add_scalar("Params/l_20", model.l_20.item(), epoch + 1)

        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {math.sqrt(loss.item())/1000:.4f}")
        # print(f"m_1: {model.m_1.item():.4f}, m_2: {model.m_2.item():.4f}, m_3: {model.m_3.item():.4f}, m_4: {model.m_4.item():.4f}")
        # real m
        # print(f"m_1: {0.5*torch.sigmoid(model.m_1).item():.4f}, m_2: {0.5*torch.sigmoid(model.m_2).item():.4f}, m_3: {0.5*torch.sigmoid(model.m_3).item():.4f}, m_4: {0.5*torch.sigmoid(model.m_4).item():.4f}")
        print(f"S_1: {model.S_1.item():.4f}, S_2: {model.S_2.item():.4f}")
        # real S
        print(f"S_1: {torch.exp(-model.S_1).item():.4f}, S_2: {torch.exp(-model.S_2).item():.4f}")
        print(f"k_3: {model.k_3.item():.4f}, k_4: {model.k_4.item():.4f}")
        print(f"l_10: {model.l_10.item():.4f}, l_20: {model.l_20.item():.4f}")

# 保存参数至内存变量
model_state_dict = model.state_dict()

In [ ]:
model_sec = MechanicsModel()

# 从内存变量中加载参数
model_sec.load_state_dict(model_state_dict)

# 继续优化，对S_1和S_2进行优化
model_sec.S_1.requires_grad = True
model_sec.S_2.requires_grad = True
model_sec.l_10.requires_grad = False
model_sec.l_20.requires_grad = False
model_sec.k_3.requires_grad = False
model_sec.k_4.requires_grad = False


# 定义优化器（Adam 优化器）
optimizer = optim.Adam(model_sec.parameters(), lr=0.01, weight_decay=1e-4)   # 使用 L2 正则化

# 训练模型
num_epochs = 50000 * 3

for epoch in tqdm(range(num_epochs)):
    # 前向传播
    P_pred = model_sec(theta1, theta2)
    loss = loss_fn(P_pred, P_actual)

    # 反向传播和优化
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # 增加对面积的约束
    with torch.no_grad():
        model_sec.S_1.clamp_(0.0001, 0.001)
        model_sec.S_2.clamp_(0.0001, 0.001)

    # 每隔 100 轮打印一次损失
    if (epoch + 1) % 10000 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {math.sqrt(loss.item())/1000:.4f}")
        print(f"S_1: {model_sec.S_1.item():.4f}, S_2: {model_sec.S_2.item():.4f}")
        print(f"k_3: {model_sec.k_3.item():.4f}, k_4: {model_sec.k_4.item():.4f}")
        print(f"l_10: {model_sec.l_10.item():.4f}, l_20: {model_sec.l_20.item():.4f}")


In [ ]:
# save
model_state_dict = model_sec.state_dict()

In [ ]:
# 再优化
model = MechanicsModel()
model.S_1.requires_grad = False
model.S_2.requires_grad = False
model.l_10.requires_grad = True
model.l_20.requires_grad = True
model.k_3.requires_grad = True
model.k_4.requires_grad = True

model.load_state_dict(model_state_dict)

# 定义损失函数（均方误差）
loss_fn = nn.MSELoss()

# 定义优化器（Adam 优化器）
optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)   # 使用 L2 正则化

# 训练模型
num_epochs = 50000
for epoch in tqdm(range(num_epochs)):
    # 前向传播
    P_pred = model(theta1, theta2)
    loss = loss_fn(P_pred, P_actual)

    # 反向传播和优化
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # 每隔 100 轮打印一次损失
    if (epoch + 1) % 10000 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {math.sqrt(loss.item())/1000:.4f}")
        print(f"S_1: {model.S_1.item():.4f}, S_2: {model.S_2.item():.4f}")
        print(f"k_3: {model.k_3.item():.4f}, k_4: {model.k_4.item():.4f}")
        print(f"l_10: {model.l_10.item():.4f}, l_20: {model.l_20.item():.4f}")

# 保存参数至内存变量
model_state_dict = model.state_dict()

In [ ]:
model.eval()
# manually set the manu-SOTA?
model.k_3 = torch.nn.parameter.Parameter(torch.tensor(437.5840, dtype=torch.float32))
model.k_4 = torch.nn.parameter.Parameter(torch.tensor(216.2475, dtype=torch.float32))
model.l_10 = torch.nn.parameter.Parameter(torch.tensor(0.1865, dtype=torch.float32))
model.l_20 = torch.nn.parameter.Parameter(torch.tensor(0.2677, dtype=torch.float32))
model.S_1 = torch.nn.parameter.Parameter(torch.tensor(0.0006, dtype=torch.float32))
model.S_2 = torch.nn.parameter.Parameter(torch.tensor(0.0006, dtype=torch.float32))

P_pred = model(theta1, theta2)
# list all experiments
for i in range(len(theta1)):
    print(f"theta1: {theta1[i]:.4f}, theta2: {theta2[i]:.4f}, P_actual: {P_actual[i]}, P_pred: {P_pred[i]}")
# 初始误差
loss = loss_fn(P_pred, P_actual)
print(f"Final Loss: {math.sqrt(loss.item())/1000:.4f}")

In [ ]:
# 输出所有参数
print(f"Estimated k3: {model.k_3.item():.4f}")
print(f"Estimated k4: {model.k_4.item():.4f}")
# print(f"Estimated m1: {model.m_1.item():.4f}")
# print(f"Estimated m2: {model.m_2.item():.4f}")
# print(f"Estimated m3: {model.m_3.item():.4f}")
# print(f"Estimated m4: {model.m_4.item():.4f}")
print(f"Estimated l10: {model.l_10.item():.4f}")
print(f"Estimated l20: {model.l_20.item():.4f}")
# print(f"Estimated S1: {model.S_1.item():.4f}")
# print(f"Estimated S2: {model.S_2.item():.4f}")

In [ ]:
# 全局优化：网格化参数选择
m_max = 0.2
l_max = 0.3
k_max = 400

m_array = torch.linspace(0.1, m_max, 5)
l_array = torch.linspace(0.1, l_max, 10)
k_array = torch.linspace(100, k_max, 10)
S_array = torch.linspace(0.0001, 0.001, 100)

loss_all = 1000000
best_params = []

m1 = torch.tensor(0.1, dtype=torch.float32)
m2 = torch.tensor(0.1, dtype=torch.float32)
m3 = torch.tensor(0.18, dtype=torch.float32)
m4 = torch.tensor(0.18, dtype=torch.float32)
l10 = torch.tensor(0.1865, dtype=torch.float32)
l20 = torch.tensor(0.2677, dtype=torch.float32)
k3 = torch.tensor(437.5840, dtype=torch.float32)
k4 = torch.tensor(216.2475, dtype=torch.float32)


# for m1 in tqdm(m_array):
#     for m2 in m_array:
#         for m3 in m_array:
#             for m4 in m_array:
#                 for l10 in l_array:
#                     for l20 in l_array:

# 在模型基础上，优化 S1, S2
for S1 in S_array:
    for S2 in S_array:
        # model = MechanicsModel()
        # model.m_1 = nn.Parameter(m1)
        # model.m_2 = nn.Parameter(m2)
        # model.m_3 = nn.Parameter(m3)
        # model.m_4 = nn.Parameter(m4)
        # model.l_10 = nn.Parameter(l10)
        # model.l_20 = nn.Parameter(l20)
        # model.k_3 = nn.Parameter(k3)
        # model.k_4 = nn.Parameter(k4)
        model.S_1 = nn.Parameter(S1)
        model.S_2 = nn.Parameter(S2)
        P_pred = model(theta1, theta2)
        loss = loss_fn(P_pred, P_actual)
        if (loss_all >  math.sqrt(loss.item())/1000):
            print(f"loss_all update: {math.sqrt(loss.item())/1000}")
            loss_all = math.sqrt(loss.item())/1000
            best_params = [m1, m2, m3, m4, l10, l20, S1, S2]
        if math.sqrt(loss.item())/1000 < 1:
            print(f"Estimated m1: {model.m_1.item():.4f}")
            print(f"Estimated m2: {model.m_2.item():.4f}")
            print(f"Estimated m3: {model.m_3.item():.4f}")
            print(f"Estimated m4: {model.m_4.item():.4f}")
            print(f"Estimated l10: {model.l_10.item():.4f}")
            print(f"Estimated l20: {model.l_20.item():.4f}")
            print(f"Estimated S1: {model.S_1.item():.4f}")
            print(f"Estimated S2: {model.S_2.item():.4f}")
            print(f"Loss: {math.sqrt(loss.item())/1000:.4f}")
            print("=========================================")

print(best_params)
